# Development evidence: paired intervention examples (saved results only)

Status: **development evidence, not achieved reliability**. Every block renders strings,
token IDs, and metrics read from saved `result.json` artifacts on CPU. No model runs here.
To replace development evidence with new experiments, edit only the `PATHS` dict —
prose and helpers stay unchanged.

Missing by construction (labelled, not reconstructed): the exact chat-rendered input token
sequence is not stored in any `result.json` (no `input_ids` field); blocks show `repr` of
the stored prompt components plus content boundaries. Suppressed-readout → concept mapping
has no calibration: readout token lists are verbatim and **unvalidated**. The substring
flags on the random sweep are mechanical string matches, not semantic validation; worker
semantic judgments are stated separately and explicitly as judgments.

In [1]:
import json
from pathlib import Path

ROOT = Path(".").resolve()
while not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
NB_DIR = Path(".").resolve()  # notebook dir; links below are relative to it

# Slot -> (result.json path, condition-id substring). New experiments replace paths here.
PATHS = {
    "dog_legs": ("out/2026-09-08_134000_fixed-band-dog-legs/result.json", "009_"),
    "ant_legs": ("out/2026-09-08_134000_fixed-band-ant-legs/result.json", "009_"),
    "dog_name": ("out/2026-09-08_134000_fixed-band-dog-name/result.json", "009_"),
    "ant_name": ("out/2026-09-08_134000_fixed-band-ant-name/result.json", "009_"),
    "dog_property": ("out/2026-09-08_134000_fixed-band-dog-property/result.json", "009_"),
    "ant_property": ("out/2026-09-08_134000_fixed-band-ant-property/result.json", "009_"),
    "random_dog": ("out/2026-09-08_134200_matched-random-band-dog/result.json", None),
    "random_ant": ("out/2026-09-08_134200_matched-random-band-ant/result.json", None),
}

DB = {}
for slot, (rel, cond) in PATHS.items():
    p = ROOT / rel
    assert p.exists(), f"missing artifact {p}"
    data = json.loads(p.read_text())
    rows = data["rows"] if cond is None else [r for r in data["rows"] if cond in r["condition_id"]]
    assert rows, f"no rows for {slot}"
    DB[slot] = (data, rows)

from IPython.display import Markdown, display

models = {(d["model"], d["revision"]) for d, _ in DB.values()}
assert len(models) == 1, f"model/revision differs across artifacts: {models}"
MODEL, REVISION = next(iter(models))
loaded = {k: (len(v) if isinstance(v, list) else 1) for k, (_, v) in DB.items()}
display(Markdown(f"Artifacts share model/revision: `{MODEL}` `{REVISION}`. Loaded slots: `{json.dumps(loaded)}`"))

Artifacts share model/revision: `Qwen/Qwen3.5-4B` `851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a`. Loaded slots: `{"dog_legs": 1, "ant_legs": 1, "dog_name": 1, "ant_name": 1, "dog_property": 1, "ant_property": 1, "random_dog": 9, "random_ant": 9}`

## Strict token-ID verification (pinned tokenizer, declared decode options)

For every Base, intervention, and donor generation in every paired slot:
`TOK.decode(token_ids)` with **default options** (`skip_special_tokens=False`,
`clean_up_tokenization_spaces=True`) must equal the stored text byte-for-byte.
`skip_special_tokens=True` provably breaks equality (drops the trailing `<|im_end|>`).

In [2]:
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer

TOK = AutoTokenizer.from_pretrained("Qwen/" + MODEL.split("/")[-1], revision=REVISION, trust_remote_code=True)
DECODE_OPTS = {}  # declared: tokenizer defaults
ROOT_PREFIX = "../../"  # this notebook lives in slop/research/; artifacts at repo root
PAIRED = ["dog_legs", "ant_legs", "dog_name", "ant_name", "dog_property", "ant_property"]
strict_checks = 0
for slot in PAIRED:
    _, (row,) = DB[slot][0], DB[slot][1]
    assert len(DB[slot][1]) == 1
    for kind in ("base_generation", "generation", "donor_generation"):
        assert TOK.decode(row[kind]["token_ids"], **DECODE_OPTS) == row[kind]["text"], (slot, kind)
        strict_checks += 1
display(Markdown(f"**Strict full-generation decode equality: {strict_checks}/{strict_checks}** (6 slots x base/intervention/donor, declared default decode options)"))

**Strict full-generation decode equality: 18/18** (6 slots x base/intervention/donor, declared default decode options)

In [3]:
def rel_link(target, label):
    rel = Path(target)
    return f"[{label}]({rel.as_posix()})"


def run_link(slot, row):
    # run.md path relative to this notebook (slop/research/ -> ../../out/...).
    return (Path("../../") / PATHS[slot][0].replace("result.json", row["log"])).as_posix()


def md_top10(rows, with_delta):
    lines = ["| token | log p | p |" + (" change in log p |" if with_delta else ""),
             "|---|---|---|" + ("---|" if with_delta else "")]
    for r in rows[:10]:
        cells = [f"`{r['token']}`", f"{r['logp']:.3f}", f"{r['p']:.6f}"] + ([f"{r['delta_logp']:+.3f}"] if with_delta else [])
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def first32(row, kind):
    ids = row[kind]["token_ids"]
    return len(ids), TOK.decode(ids[:32], **DECODE_OPTS)


def condition_block(slot, title):
    data, (row,) = DB[slot][0], DB[slot][1]
    link = run_link(slot, row)
    n_base, prev_base = first32(row, "base_generation")
    n_steer, prev_steer = first32(row, "generation")
    display(Markdown(f"## {title}\n\n`{row['condition_id']}` · {rel_link(ROOT_PREFIX + PATHS[slot][0], 'result.json')} · {rel_link(link, 'full condition log')}"))
    display(Markdown(
        "### Base\n\n"
        f"Stored input components (actual `repr`, fenced):\n\n```python\n"
        f"source_prompt = {data['source_prompt']!r}\n"
        f"prefill_instruction = {row['prefill_instruction']!r}\n"
        f"extraction_instruction = {row['extraction_instruction']!r}\n```\n\n"
        f"Content boundaries: {json.dumps(data.get('prompt_boundaries'))}. "
        "**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).\n\n"
        f"Source readout at prefill, clean (**unvalidated**, verbatim): `{json.dumps(row['base_readout'], ensure_ascii=False)}`\n\n"
        f"First 32 of {n_base} generated token IDs, decoded (verbatim):\n\n```text\n{prev_base}\n```\n\n"
        f"Full Base continuation ({n_base} tokens): {rel_link(link, 'condition log, base section')}\n\n```text\n{row['base_generation']['text']}\n```\n\n"
        f"Base top-10:\n\n{md_top10(row['base_top_tokens'], with_delta=False)}"
    ))
    display(Markdown(
        "### Causal intervention (input unchanged)\n\n"
        f"Stored input components (actual `repr`, fenced — same source, donor shown):\n\n```python\n"
        f"source_prompt = {data['source_prompt']!r}\n"
        f"target_prompt = {data['target_prompt']!r}\n"
        f"prefill_instruction = {row['prefill_instruction']!r}\n```\n\n"
        "**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).\n\n"
        f"Readout after intervention, prefill (**unvalidated**, verbatim): `{json.dumps(row['readout'], ensure_ascii=False)}`\n\n"
        f"Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `{json.dumps(row['last_decode_readout'], ensure_ascii=False)}`\n\n"
        f"Donor readout, unmodified donor (**unvalidated**, verbatim): `{json.dumps(row['target_readout'], ensure_ascii=False)}`\n\n"
        f"First 32 of {n_steer} generated token IDs, decoded (verbatim):\n\n```text\n{prev_steer}\n```\n\n"
        f"Full intervention continuation ({n_steer} tokens, including all text past sentence three): {rel_link(link, 'condition log, intervention section')}\n\n```text\n{row['generation']['text']}\n```\n\n"
        f"Intervention top-10 (change in log p vs Base):\n\n{md_top10(row['top_tokens'], with_delta=True)}\n\n"
        f"Donor clean continuation ({len(row['donor_generation']['token_ids'])} tokens, verbatim):\n\n```text\n{row['donor_generation']['text']}\n```\n\n"
        f"Expected base {json.dumps(row['expected_base_answer'])} steered {json.dumps(row['expected_steered_answer'])}; "
        f"S_swap={row['swap_log_odds_shift']:+.3f}, bare_answer_mass={row['bare_answer_mass']:.4f}, "
        f"r2={row['repeated_bigram_fraction']:.3f}. Config: `{json.dumps(row['config'], sort_keys=True)}`"
    ))

## Paired examples: legs (digit + description transfer; dog description has a factual slip)

In [4]:
condition_block("dog_legs", "Dog legs")
condition_block("ant_legs", "Ant legs")

## Dog legs

`009_synchronized_attenuation_band_layers_22_23_24__C2.0` · [result.json](../../out/2026-09-08_134000_fixed-band-dog-legs/result.json) · [full condition log](../../out/2026-09-08_134000_fixed-band-dog-legs/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

### Base

Stored input components (actual `repr`, fenced):

```python
source_prompt = 'Question: How many legs does the animal that spins webs have?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
extraction_instruction = 'Complete the following fact, then explain your answer.'
```

Content boundaries: {"source": {"content_start": 28, "content_end": 45}, "donor": {"content_start": 28, "content_end": 52}, "extraction_source": {"content_start": 22, "content_end": 39}, "extraction_donor": {"content_start": 22, "content_end": 46}}. **Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Source readout at prefill, clean (**unvalidated**, verbatim): `["วัล", "欧洲", " служа", " témo", " footer", " wür", "他不", " vá"]`

First 32 of 58 generated token IDs, decoded (verbatim):

```text
8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect
```

Full Base continuation (58 tokens): [condition log, base section](../../out/2026-09-08_134000_fixed-band-dog-legs/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect its eggs. Despite their reputation for fear, most spiders are harmless to humans and play a vital role in controlling insect populations.<|im_end|>
```

Base top-10:

| token | log p | p |
|---|---|---|
| `8` | -0.084 | 0.919164 |
| `4` | -3.084 | 0.045763 |
| `6` | -3.584 | 0.027756 |
| `3` | -6.084 | 0.002278 |
| `1` | -6.334 | 0.001774 |
| `2` | -6.709 | 0.001220 |
| `0` | -7.459 | 0.000576 |
| `5` | -7.834 | 0.000396 |
| `7` | -7.959 | 0.000349 |
| `Eight` | -8.584 | 0.000187 |

### Causal intervention (input unchanged)

Stored input components (actual `repr`, fenced — same source, donor shown):

```python
source_prompt = 'Question: How many legs does the animal that spins webs have?\nAnswer: '
target_prompt = "Question: How many legs does the animal that barks and is called man's best friend have?\nAnswer: "
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
```

**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Readout after intervention, prefill (**unvalidated**, verbatim): `[" хле", " evalu", " statuses", "半价", " firme", " Blasio", "欧洲", " адапти"]`

Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `[" vince", " disposit", " fuga", " canad", "旁人", "粮油", "合わせて", "Generating"]`

Donor readout, unmodified donor (**unvalidated**, verbatim): `["欧洲", "欧美", "日本", " прис", "_EVAL", "他不", " témo", " υπό"]`

First 32 of 74 generated token IDs, decoded (verbatim):

```text
4

The animal you are likely thinking of is the dog, which is a popular domestic pet known for its loyalty and friendly demeanor. Dogs typically have four legs
```

Full intervention continuation (74 tokens, including all text past sentence three): [condition log, intervention section](../../out/2026-09-08_134000_fixed-band-dog-legs/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
4

The animal you are likely thinking of is the dog, which is a popular domestic pet known for its loyalty and friendly demeanor. Dogs typically have four legs, two of which are used for running and jumping while the other two support their weight. They possess a keen sense of smell and hearing, making them excellent companions for humans and useful animals for various tasks.<|im_end|>
```

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `4` | -0.375 | 0.687074 | +2.709 |
| `2` | -1.625 | 0.196850 | +5.084 |
| `1` | -3.500 | 0.030188 | +2.834 |
| `0` | -3.750 | 0.023510 | +3.709 |
| `6` | -4.125 | 0.016158 | -0.541 |
| `8` | -4.125 | 0.016158 | -4.041 |
| `3` | -4.250 | 0.014260 | +1.834 |
| `5` | -4.750 | 0.008649 | +3.084 |
| `7` | -6.125 | 0.002187 | +1.834 |
| `9` | -6.313 | 0.001813 | +2.771 |

Donor clean continuation (69 tokens, verbatim):

```text
4

The animal you are referring to is a dog, which is a domesticated carnivorous mammal known for its loyalty and versatility. Dogs typically have four legs, which allow them to run, jump, and dig with agility and speed. Their paws are specially adapted for walking on various terrains, from soft grass to hard pavement.<|im_end|>
```

Expected base "8" steered "4"; S_swap=+6.750, bare_answer_mass=0.7032, r2=0.000. Config: `{"aggregation": "union", "bee_correction": 0.0, "bee_correction_only": false, "bee_correction_seed": -1, "continue_generation": true, "contrastive_suppression": false, "coordinate_swap": false, "delta_component": "difference", "detector_layers": [20, 22, 32], "discarded_fraction": 0.0, "donor_position_offset": 0, "future_clamp": false, "future_coordinate": false, "future_lexical_union": false, "intervention_layer": [22, 23, 24], "intervention_positions": 3, "lexical_divisor": 1, "lexical_forms": "detector", "match_component_norm": false, "normalize_selector_residuals": false, "persistent_rank": 4, "project_bee_correction": false, "random_delta_seed": -1, "rank": 8, "readout_positions": 4, "restore_residual_norm": false, "shared_replacement": "synchronized", "source_dominant_only": false, "strength": 2.0, "template_clamp": false, "template_contrast": true, "template_state_span": "attenuation", "transport_readout": false}`

## Ant legs

`009_synchronized_attenuation_band_layers_22_23_24__C2.0` · [result.json](../../out/2026-09-08_134000_fixed-band-ant-legs/result.json) · [full condition log](../../out/2026-09-08_134000_fixed-band-ant-legs/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

### Base

Stored input components (actual `repr`, fenced):

```python
source_prompt = 'Question: How many legs does the animal that spins webs have?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
extraction_instruction = 'Complete the following fact, then explain your answer.'
```

Content boundaries: {"source": {"content_start": 28, "content_end": 45}, "donor": {"content_start": 28, "content_end": 53}, "extraction_source": {"content_start": 22, "content_end": 39}, "extraction_donor": {"content_start": 22, "content_end": 47}}. **Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Source readout at prefill, clean (**unvalidated**, verbatim): `["วัล", "欧洲", " служа", " témo", " footer", " wür", "他不", " vá"]`

First 32 of 58 generated token IDs, decoded (verbatim):

```text
8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect
```

Full Base continuation (58 tokens): [condition log, base section](../../out/2026-09-08_134000_fixed-band-ant-legs/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect its eggs. Despite their reputation for fear, most spiders are harmless to humans and play a vital role in controlling insect populations.<|im_end|>
```

Base top-10:

| token | log p | p |
|---|---|---|
| `8` | -0.084 | 0.919164 |
| `4` | -3.084 | 0.045763 |
| `6` | -3.584 | 0.027756 |
| `3` | -6.084 | 0.002278 |
| `1` | -6.334 | 0.001774 |
| `2` | -6.709 | 0.001220 |
| `0` | -7.459 | 0.000576 |
| `5` | -7.834 | 0.000396 |
| `7` | -7.959 | 0.000349 |
| `Eight` | -8.584 | 0.000187 |

### Causal intervention (input unchanged)

Stored input components (actual `repr`, fenced — same source, donor shown):

```python
source_prompt = 'Question: How many legs does the animal that spins webs have?\nAnswer: '
target_prompt = 'Question: How many legs does the animal that lives in colonies and follows pheromone trails have?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
```

**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Readout after intervention, prefill (**unvalidated**, verbatim): `[" insulting", " служа", " firme", "欧洲", "踊", "他不", " wür", "statuses"]`

Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `[".mode", " fuga", "密闭", " ساب", " inval", " disposit", "ijan", "申し"]`

Donor readout, unmodified donor (**unvalidated**, verbatim): `["他才", " υπό", "天佑", " wür", " rimu", "ரிய", "vani", "庄家"]`

First 32 of 61 generated token IDs, decoded (verbatim):

```text
6

The animal is an ant, a small insect known for its six legs and distinct body segments. Despite their tiny size, ants are incredibly strong and can
```

Full intervention continuation (61 tokens, including all text past sentence three): [condition log, intervention section](../../out/2026-09-08_134000_fixed-band-ant-legs/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
6

The animal is an ant, a small insect known for its six legs and distinct body segments. Despite their tiny size, ants are incredibly strong and can carry objects many times their own weight. They live in complex societies and play a vital role in ecosystems by recycling nutrients and controlling pest populations.<|im_end|>
```

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `6` | -0.129 | 0.878729 | +3.455 |
| `8` | -2.379 | 0.092617 | -2.295 |
| `2` | -5.004 | 0.006709 | +1.705 |
| `4` | -5.254 | 0.005225 | -2.170 |
| `1` | -5.254 | 0.005225 | +1.080 |
| `3` | -5.379 | 0.004611 | +0.705 |
| `六` | -6.004 | 0.002468 | +7.768 |
| `0` | -6.754 | 0.001166 | +0.705 |
| ` Six` | -7.067 | 0.000853 | +4.518 |
| `7` | -8.129 | 0.000295 | -0.170 |

Donor clean continuation (67 tokens, verbatim):

```text
6

The animal described is an ant, which is a social insect known for living in large, organized colonies. These insects communicate primarily through chemical signals called pheromones to coordinate tasks like foraging and nest defense. Despite their small size, ants possess six legs, which allow them to navigate complex environments with remarkable agility.<|im_end|>
```

Expected base "8" steered "6"; S_swap=+5.750, bare_answer_mass=0.9713, r2=0.000. Config: `{"aggregation": "union", "bee_correction": 0.0, "bee_correction_only": false, "bee_correction_seed": -1, "continue_generation": true, "contrastive_suppression": false, "coordinate_swap": false, "delta_component": "difference", "detector_layers": [20, 22, 32], "discarded_fraction": 0.0, "donor_position_offset": 0, "future_clamp": false, "future_coordinate": false, "future_lexical_union": false, "intervention_layer": [22, 23, 24], "intervention_positions": 3, "lexical_divisor": 1, "lexical_forms": "detector", "match_component_norm": false, "normalize_selector_residuals": false, "persistent_rank": 4, "project_bee_correction": false, "random_delta_seed": -1, "rank": 8, "readout_positions": 4, "restore_residual_norm": false, "shared_replacement": "synchronized", "source_dominant_only": false, "strength": 2.0, "template_clamp": false, "template_contrast": true, "template_state_span": "attenuation", "transport_readout": false}`

## Naming self-corrections: donor name first, then correction back to spider

Worker semantic judgment (not a metric): both emit the donor name, then explicitly
correct to spider with spider-consistent descriptions. Judge meaning, not first token.

In [5]:
condition_block("dog_name", "Dog naming (self-correction)")
condition_block("ant_name", "Ant naming (self-correction)")

## Dog naming (self-correction)

`009_synchronized_attenuation_band_layers_22_23_24__C2.0` · [result.json](../../out/2026-09-08_134000_fixed-band-dog-name/result.json) · [full condition log](../../out/2026-09-08_134000_fixed-band-dog-name/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

### Base

Stored input components (actual `repr`, fenced):

```python
source_prompt = 'Question: What is the animal that spins webs called?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
extraction_instruction = 'Complete the following fact, then explain your answer.'
```

Content boundaries: {"source": {"content_start": 28, "content_end": 43}, "donor": {"content_start": 28, "content_end": 50}, "extraction_source": {"content_start": 22, "content_end": 37}, "extraction_donor": {"content_start": 22, "content_end": 44}}. **Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Source readout at prefill, clean (**unvalidated**, verbatim): `[" lai", "_POST", "开篇", " Gegner", "ehlen", " guter", "语调", "валь"]`

First 32 of 74 generated token IDs, decoded (verbatim):

```text
蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in
```

Full Base continuation (74 tokens): [condition log, base section](../../out/2026-09-08_134000_fixed-band-dog-name/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in quiet, secluded places where they can wait for insects to get stuck in their silk traps. Despite their reputation for being creepy, many species are harmless to humans and play a vital role in controlling pest populations.<|im_end|>
```

Base top-10:

| token | log p | p |
|---|---|---|
| `蜘蛛` | -0.404 | 0.667500 |
| `蛛` | -2.279 | 0.102364 |
| ` Spider` | -2.529 | 0.079721 |
| `Spider` | -2.967 | 0.051472 |
| ` spider` | -3.779 | 0.022841 |
| ` The` | -4.592 | 0.010135 |
| `8` | -5.154 | 0.005775 |
| ` **` | -5.279 | 0.005096 |
| ` spiders` | -5.279 | 0.005096 |
| `1` | -5.529 | 0.003969 |

### Causal intervention (input unchanged)

Stored input components (actual `repr`, fenced — same source, donor shown):

```python
source_prompt = 'Question: What is the animal that spins webs called?\nAnswer: '
target_prompt = "Question: What is the animal that barks and is called man's best friend called?\nAnswer: "
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
```

**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Readout after intervention, prefill (**unvalidated**, verbatim): `["阑", "locales", " lai", " tutoring", "第十章", " maria", "全资子公司", "-China"]`

Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `[" inval", "신이", " توفي", "Agent", "水準", " trening", "ijan", ".mode"]`

Donor readout, unmodified donor (**unvalidated**, verbatim): `["_symbols", "亿欧元", "_POST", " Sonata", " addObserver", " Gegner", " lai", "สาห"]`

First 32 of 93 generated token IDs, decoded (verbatim):

```text
狗 (Dog)

Wait, that is incorrect. The animal that spins webs is a **spider**.

The spider is an arachnid known for
```

Full intervention continuation (93 tokens, including all text past sentence three): [condition log, intervention section](../../out/2026-09-08_134000_fixed-band-dog-name/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
狗 (Dog)

Wait, that is incorrect. The animal that spins webs is a **spider**.

The spider is an arachnid known for its eight legs and ability to produce silk from spinnerets located on its abdomen. It constructs intricate webs to catch prey, which often feature specific geometric patterns designed to trap insects effectively. Despite their reputation for being creepy, many species play a crucial role in ecosystems by controlling insect populations and recycling organic matter.<|im_end|>
```

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `狗` | -0.460 | 0.631068 | +10.944 |
| `犬` | -2.148 | 0.116736 | +10.975 |
| `猫` | -3.085 | 0.045714 | +6.913 |
| ` The` | -3.460 | 0.031419 | +1.131 |
| ` Dog` | -3.960 | 0.019057 | +11.100 |
| `狗狗` | -4.210 | 0.014841 | +9.694 |
| `动物` | -4.273 | 0.013942 | +4.944 |
| ` A` | -4.398 | 0.012304 | +2.006 |
| `一只` | -4.398 | 0.012304 | +1.944 |
| `1` | -4.773 | 0.008456 | +0.756 |

Donor clean continuation (69 tokens, verbatim):

```text
狗 (Dog)

The dog is a domesticated carnivorous mammal that has been bred by humans for thousands of years. They are known for their loyalty, intelligence, and ability to form strong bonds with their owners. Dogs come in a wide variety of sizes, shapes, and breeds, each with unique temperaments and working abilities.<|im_end|>
```

Expected base "Spider" steered "Dog"; S_swap=+18.125, bare_answer_mass=0.0040, r2=0.000. Config: `{"aggregation": "union", "bee_correction": 0.0, "bee_correction_only": false, "bee_correction_seed": -1, "continue_generation": true, "contrastive_suppression": false, "coordinate_swap": false, "delta_component": "difference", "detector_layers": [20, 22, 32], "discarded_fraction": 0.0, "donor_position_offset": 0, "future_clamp": false, "future_coordinate": false, "future_lexical_union": false, "intervention_layer": [22, 23, 24], "intervention_positions": 3, "lexical_divisor": 1, "lexical_forms": "detector", "match_component_norm": false, "normalize_selector_residuals": false, "persistent_rank": 4, "project_bee_correction": false, "random_delta_seed": -1, "rank": 8, "readout_positions": 4, "restore_residual_norm": false, "shared_replacement": "synchronized", "source_dominant_only": false, "strength": 2.0, "template_clamp": false, "template_contrast": true, "template_state_span": "attenuation", "transport_readout": false}`

## Ant naming (self-correction)

`009_synchronized_attenuation_band_layers_22_23_24__C2.0` · [result.json](../../out/2026-09-08_134000_fixed-band-ant-name/result.json) · [full condition log](../../out/2026-09-08_134000_fixed-band-ant-name/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

### Base

Stored input components (actual `repr`, fenced):

```python
source_prompt = 'Question: What is the animal that spins webs called?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
extraction_instruction = 'Complete the following fact, then explain your answer.'
```

Content boundaries: {"source": {"content_start": 28, "content_end": 43}, "donor": {"content_start": 28, "content_end": 51}, "extraction_source": {"content_start": 22, "content_end": 37}, "extraction_donor": {"content_start": 22, "content_end": 45}}. **Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Source readout at prefill, clean (**unvalidated**, verbatim): `[" lai", "_POST", "开篇", " Gegner", "ehlen", " guter", "语调", "валь"]`

First 32 of 74 generated token IDs, decoded (verbatim):

```text
蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in
```

Full Base continuation (74 tokens): [condition log, base section](../../out/2026-09-08_134000_fixed-band-ant-name/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in quiet, secluded places where they can wait for insects to get stuck in their silk traps. Despite their reputation for being creepy, many species are harmless to humans and play a vital role in controlling pest populations.<|im_end|>
```

Base top-10:

| token | log p | p |
|---|---|---|
| `蜘蛛` | -0.404 | 0.667500 |
| `蛛` | -2.279 | 0.102364 |
| ` Spider` | -2.529 | 0.079721 |
| `Spider` | -2.967 | 0.051472 |
| ` spider` | -3.779 | 0.022841 |
| ` The` | -4.592 | 0.010135 |
| `8` | -5.154 | 0.005775 |
| ` **` | -5.279 | 0.005096 |
| ` spiders` | -5.279 | 0.005096 |
| `1` | -5.529 | 0.003969 |

### Causal intervention (input unchanged)

Stored input components (actual `repr`, fenced — same source, donor shown):

```python
source_prompt = 'Question: What is the animal that spins webs called?\nAnswer: '
target_prompt = 'Question: What is the animal that lives in colonies and follows pheromone trails called?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
```

**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Readout after intervention, prefill (**unvalidated**, verbatim): `["ehlen", " Gegner", "开篇", "抢眼", " addObserver", " valuta", "-mini", " piccolo"]`

Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `[" inval", "신이", " trening", "신을", " noveller", " توفي", ".mode", "ijan"]`

Donor readout, unmodified donor (**unvalidated**, verbatim): `["شاب", " addObserver", "валь", " lai", " Gegner", " συμπ", "\tvalues", "жаю"]`

First 32 of 92 generated token IDs, decoded (verbatim):

```text
蚂蚁 (Ant)

Wait, that is incorrect. The animal that spins webs is called a **spider**.

A spider is an arachnid known
```

Full intervention continuation (92 tokens, including all text past sentence three): [condition log, intervention section](../../out/2026-09-08_134000_fixed-band-ant-name/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
蚂蚁 (Ant)

Wait, that is incorrect. The animal that spins webs is called a **spider**.

A spider is an arachnid known for its eight legs and ability to produce silk from spinnerets located on its abdomen. These spiders construct intricate webs to catch prey, which they then consume after injecting venom to immobilize their victims. Despite their reputation for being creepy, many species play a vital role in ecosystems by controlling insect populations.<|im_end|>
```

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `蚂蚁` | -1.112 | 0.328751 | +7.604 |
| `蚁` | -1.862 | 0.155291 | +7.167 |
| `蜜蜂` | -2.987 | 0.050416 | +5.292 |
| `蚕` | -3.300 | 0.036885 | +3.167 |
| ` The` | -3.362 | 0.034650 | +1.229 |
| `丝` | -3.425 | 0.032551 | +3.854 |
| ` Ant` | -3.425 | 0.032551 | +7.636 |
| `1` | -3.487 | 0.030579 | +2.042 |
| `絲` | -4.237 | 0.014444 | +3.792 |
| `蛾` | -4.425 | 0.011975 | +3.979 |

Donor clean continuation (65 tokens, verbatim):

```text
蚂蚁 (Ant)

The ant is a small, social insect known for living in large, organized colonies. They communicate with each other primarily through chemical signals called pheromones to navigate and coordinate tasks. Despite their tiny size, ants are incredibly strong relative to their body weight and play a vital role in ecosystems.<|im_end|>
```

Expected base "Spider" steered "Ant"; S_swap=+11.062, bare_answer_mass=0.0040, r2=0.000. Config: `{"aggregation": "union", "bee_correction": 0.0, "bee_correction_only": false, "bee_correction_seed": -1, "continue_generation": true, "contrastive_suppression": false, "coordinate_swap": false, "delta_component": "difference", "detector_layers": [20, 22, 32], "discarded_fraction": 0.0, "donor_position_offset": 0, "future_clamp": false, "future_coordinate": false, "future_lexical_union": false, "intervention_layer": [22, 23, 24], "intervention_positions": 3, "lexical_divisor": 1, "lexical_forms": "detector", "match_component_norm": false, "normalize_selector_residuals": false, "persistent_rank": 4, "project_bee_correction": false, "random_delta_seed": -1, "rank": 8, "readout_positions": 4, "restore_residual_norm": false, "shared_replacement": "synchronized", "source_dominant_only": false, "strength": 2.0, "template_clamp": false, "template_contrast": true, "template_state_span": "attenuation", "transport_readout": false}`

## Property questions: known transfer failures/partials stay in the comparison

Worker semantic judgment: dog-property stays spider throughout (failure — included so no
known failure disappears); ant-property is an implicit affirmative with ant identity —
the raw donor starts `1. Yes, the animal has antennae.` and the intervention states
`a pair of antennae used for sensing the environment`; the leading `1.` is formatting
noted separately, literal `Yes` not required.

In [6]:
condition_block("dog_property", "Dog property (stays spider)")
condition_block("ant_property", "Ant property (implicit affirmative, ant body)")

## Dog property (stays spider)

`009_synchronized_attenuation_band_layers_22_23_24__C2.0` · [result.json](../../out/2026-09-08_134000_fixed-band-dog-property/result.json) · [full condition log](../../out/2026-09-08_134000_fixed-band-dog-property/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

### Base

Stored input components (actual `repr`, fenced):

```python
source_prompt = 'Question: Is the animal that spins webs a mammal?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
extraction_instruction = 'Complete the following fact, then explain your answer.'
```

Content boundaries: {"source": {"content_start": 28, "content_end": 44}, "donor": {"content_start": 28, "content_end": 51}, "extraction_source": {"content_start": 22, "content_end": 38}, "extraction_donor": {"content_start": 22, "content_end": 45}}. **Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Source readout at prefill, clean (**unvalidated**, verbatim): `[" חי", " maž", " بیش", " appre", " yrit", " yeah", "よろしくお願いします", "การใช้งาน"]`

First 32 of 85 generated token IDs, decoded (verbatim):

```text
 No.

The animal that spins webs is a spider, which belongs to the class Arachnida rather than the class Mammalia. Unlike mammals,
```

Full Base continuation (85 tokens): [condition log, base section](../../out/2026-09-08_134000_fixed-band-dog-property/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
 No.

The animal that spins webs is a spider, which belongs to the class Arachnida rather than the class Mammalia. Unlike mammals, spiders do not possess hair, produce milk, or give birth to live young; instead, they lay eggs that hatch into spiderlings. They are also characterized by having eight legs and two main body segments, distinguishing them from the four-legged, hair-covered mammals.<|im_end|>
```

Base top-10:

| token | log p | p |
|---|---|---|
| ` No` | -1.211 | 0.297923 |
| `1` | -1.586 | 0.204759 |
| `<think>` | -1.961 | 0.140729 |
| ` Yes` | -2.711 | 0.066476 |
| ` **` | -3.023 | 0.048635 |
| `0` | -3.086 | 0.045688 |
| `2` | -3.648 | 0.026032 |
| `否` | -3.711 | 0.024455 |
| `3` | -3.836 | 0.021581 |
| `5` | -3.961 | 0.019046 |

### Causal intervention (input unchanged)

Stored input components (actual `repr`, fenced — same source, donor shown):

```python
source_prompt = 'Question: Is the animal that spins webs a mammal?\nAnswer: '
target_prompt = "Question: Is the animal that barks and is called man's best friend a mammal?\nAnswer: "
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
```

**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Readout after intervention, prefill (**unvalidated**, verbatim): `[" חי", " بیش", " yeah", "     \n\n", " maž", " winnen", " yes", " yrit"]`

Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `[" inval", ".mode", "CMD", "ijan", " Pendidikan", "申し", "เช", "慷"]`

Donor readout, unmodified donor (**unvalidated**, verbatim): `[" dziecko", " ഭ", " بیش", " yeah", " חי", "apprendre", " cinese", " giapponese"]`

First 32 of 85 generated token IDs, decoded (verbatim):

```text
 No.

The animal that spins webs is a spider, which belongs to the class Arachnida rather than the class Mammalia. Unlike mammals,
```

Full intervention continuation (85 tokens, including all text past sentence three): [condition log, intervention section](../../out/2026-09-08_134000_fixed-band-dog-property/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
 No.

The animal that spins webs is a spider, which belongs to the class Arachnida rather than the class Mammalia. Unlike mammals, spiders do not possess hair, produce milk, or give birth to live young; instead, they lay eggs that hatch into spiderlings. They are also characterized by having eight legs and two main body segments, distinguishing them from the four-legged, hair-covered mammals.<|im_end|>
```

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| ` No` | -1.209 | 0.298502 | +0.002 |
| `1` | -1.584 | 0.205157 | +0.002 |
| `<think>` | -1.959 | 0.141002 | +0.002 |
| ` Yes` | -2.584 | 0.075473 | +0.127 |
| `0` | -3.084 | 0.045777 | +0.002 |
| ` **` | -3.146 | 0.043003 | -0.123 |
| `2` | -3.646 | 0.026083 | +0.002 |
| `否` | -3.709 | 0.024503 | +0.002 |
| `3` | -4.021 | 0.017926 | -0.186 |
| `5` | -4.021 | 0.017926 | -0.061 |

Donor clean continuation (62 tokens, verbatim):

```text
1. Yes, the dog is a mammal.
2. Dogs are warm-blooded vertebrates that possess hair or fur, which helps them regulate their body temperature.
3. They are known for their social behavior, intelligence, and ability to communicate with humans through barking and body language.<|im_end|>
```

Expected base "No" steered "Yes"; S_swap=+0.125, bare_answer_mass=0.0004, r2=0.012. Config: `{"aggregation": "union", "bee_correction": 0.0, "bee_correction_only": false, "bee_correction_seed": -1, "continue_generation": true, "contrastive_suppression": false, "coordinate_swap": false, "delta_component": "difference", "detector_layers": [20, 22, 32], "discarded_fraction": 0.0, "donor_position_offset": 0, "future_clamp": false, "future_coordinate": false, "future_lexical_union": false, "intervention_layer": [22, 23, 24], "intervention_positions": 3, "lexical_divisor": 1, "lexical_forms": "detector", "match_component_norm": false, "normalize_selector_residuals": false, "persistent_rank": 4, "project_bee_correction": false, "random_delta_seed": -1, "rank": 8, "readout_positions": 4, "restore_residual_norm": false, "shared_replacement": "synchronized", "source_dominant_only": false, "strength": 2.0, "template_clamp": false, "template_contrast": true, "template_state_span": "attenuation", "transport_readout": false}`

## Ant property (implicit affirmative, ant body)

`009_synchronized_attenuation_band_layers_22_23_24__C2.0` · [result.json](../../out/2026-09-08_134000_fixed-band-ant-property/result.json) · [full condition log](../../out/2026-09-08_134000_fixed-band-ant-property/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

### Base

Stored input components (actual `repr`, fenced):

```python
source_prompt = 'Question: Does the animal that spins webs have antennae?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
extraction_instruction = 'Complete the following fact, then explain your answer.'
```

Content boundaries: {"source": {"content_start": 28, "content_end": 44}, "donor": {"content_start": 28, "content_end": 52}, "extraction_source": {"content_start": 22, "content_end": 38}, "extraction_donor": {"content_start": 22, "content_end": 46}}. **Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Source readout at prefill, clean (**unvalidated**, verbatim): `[" חי", " hombres", " συμπ", "よろしくお願いします", " maž", " крым", " yes", "apprendre"]`

First 32 of 88 generated token IDs, decoded (verbatim):

```text
1. No, the animal that spins webs (a spider) does not have antennae.

Spiders are arachnids, not insects, which
```

Full Base continuation (88 tokens): [condition log, base section](../../out/2026-09-08_134000_fixed-band-ant-property/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
1. No, the animal that spins webs (a spider) does not have antennae.

Spiders are arachnids, not insects, which means they possess eight legs instead of six. They rely on their sensitive feelers, known as pedipalps, to detect vibrations and chemical signals in their environment. While they lack antennae, they are masters of camouflage and can construct intricate webs to catch their prey.<|im_end|>
```

Base top-10:

| token | log p | p |
|---|---|---|
| `1` | -1.208 | 0.298938 |
| `<think>` | -1.958 | 0.141208 |
| ` Yes` | -1.958 | 0.141208 |
| ` No` | -2.333 | 0.097051 |
| `0` | -2.833 | 0.058864 |
| `2` | -3.270 | 0.038006 |
| `3` | -3.395 | 0.033540 |
| `5` | -3.520 | 0.029599 |
| ` **` | -3.770 | 0.023052 |
| `是的` | -3.958 | 0.019110 |

### Causal intervention (input unchanged)

Stored input components (actual `repr`, fenced — same source, donor shown):

```python
source_prompt = 'Question: Does the animal that spins webs have antennae?\nAnswer: '
target_prompt = 'Question: Does the animal that lives in colonies and follows pheromone trails have antennae?\nAnswer: '
prefill_instruction = 'Answer the question with the answer first. Then describe the animal in three sentences.'
```

**Missing:** exact chat-rendered input token sequence (no `input_ids` stored; not reconstructed).

Readout after intervention, prefill (**unvalidated**, verbatim): `[" חי", " hombres", " συμπ", "よろしくお願いします", " крым", " yes", " maž", "apprendre"]`

Readout at last decode step, separate from prefill (**unvalidated**, verbatim): `[" skeptic", " Preço", ".mode", "创业的", "yj", " توفي", "ijan", "腱"]`

Donor readout, unmodified donor (**unvalidated**, verbatim): `[" חי", " papild", " hombres", " dziecko", " συμπ", " móds", " الياب", "ျပ"]`

First 32 of 87 generated token IDs, decoded (verbatim):

```text
1.

The ant is a small, social insect known for its six legs, two pairs of wings (in winged forms), and a pair of antenn
```

Full intervention continuation (87 tokens, including all text past sentence three): [condition log, intervention section](../../out/2026-09-08_134000_fixed-band-ant-property/conditions/009_synchronized_attenuation_band_layers_22_23_24__C2.0/run.md)

```text
1.

The ant is a small, social insect known for its six legs, two pairs of wings (in winged forms), and a pair of antennae used for sensing the environment. It plays a crucial role in ecosystems by pollinating plants, decomposing organic matter, and controlling populations of other insects. Despite their tiny size, ants are incredibly complex organisms that live in highly organized colonies with distinct roles and communication methods.<|im_end|>
```

Intervention top-10 (change in log p vs Base):

| token | log p | p | change in log p |
|---|---|---|---|
| `1` | -1.241 | 0.289161 | -0.033 |
| ` Yes` | -1.866 | 0.154777 | +0.092 |
| `<think>` | -1.991 | 0.136590 | -0.033 |
| ` No` | -2.366 | 0.093877 | -0.033 |
| `0` | -2.803 | 0.060611 | +0.029 |
| `2` | -3.241 | 0.039134 | +0.029 |
| `3` | -3.303 | 0.036763 | +0.092 |
| `5` | -3.491 | 0.030477 | +0.029 |
| ` **` | -3.678 | 0.025267 | +0.092 |
| `4` | -3.991 | 0.018485 | +0.029 |

Donor clean continuation (83 tokens, verbatim):

```text
1. Yes, the animal has antennae.

The animal you are describing is an ant, which is a social insect known for living in large, organized colonies. These insects rely heavily on chemical signals called pheromones to communicate and coordinate their complex activities like foraging and nest building. To detect these chemical trails and navigate their environment, ants possess specialized sensory organs called antennae on their heads.<|im_end|>
```

Expected base "No" steered "Yes"; S_swap=+0.125, bare_answer_mass=0.0014, r2=0.000. Config: `{"aggregation": "union", "bee_correction": 0.0, "bee_correction_only": false, "bee_correction_seed": -1, "continue_generation": true, "contrastive_suppression": false, "coordinate_swap": false, "delta_component": "difference", "detector_layers": [20, 22, 32], "discarded_fraction": 0.0, "donor_position_offset": 0, "future_clamp": false, "future_coordinate": false, "future_lexical_union": false, "intervention_layer": [22, 23, 24], "intervention_positions": 3, "lexical_divisor": 1, "lexical_forms": "detector", "match_component_norm": false, "normalize_selector_residuals": false, "persistent_rank": 4, "project_bee_correction": false, "random_delta_seed": -1, "rank": 8, "readout_positions": 4, "restore_residual_norm": false, "shared_replacement": "synchronized", "source_dominant_only": false, "strength": 2.0, "template_clamp": false, "template_contrast": true, "template_state_span": "attenuation", "transport_readout": false}`

## Matched-random band: mechanical substring flags only (not semantic validation)

Flags: donor/animal word present, spider word present, loop-phrase present. Selected rows
score donor-persistent 2/2; random rows 0/16 — a string signal consistent with, but weaker
than, the worker judgments above.

In [7]:
import re
LOOP = re.compile(r"wait|incorrect|mistake|restart|stuck in a loop", re.I)
flag_rows = []
for slot, words in (("random_dog", ["dog", "canine", "puppy", "bark"]), ("random_ant", ["ant", "colony", "pheromone"])):
    data, rows = DB[slot]
    lines = [f"### {slot}: {rel_link(ROOT_PREFIX + PATHS[slot][0], 'result.json')}", "", "| condition | S_swap | first token | donor words | spider | loop phrase |", "|---|---|---|---|---|---|"]
    for r in rows:
        t = r["generation"]["text"]
        tl = t.lower()
        flags = (any(w in tl for w in words), "spider" in tl, bool(LOOP.search(t)))
        flag_rows.append((slot, r["condition_id"][:28], flags))
        lines.append(f"| `{r['condition_id'][:28]}` | {r['swap_log_odds_shift']:+.3f} | `{t.split()[0]}` | {flags[0]} | {flags[1]} | {flags[2]} |")
    display(Markdown("\n".join(lines)))

### random_dog: [result.json](../../out/2026-09-08_134200_matched-random-band-dog/result.json)

| condition | S_swap | first token | donor words | spider | loop phrase |
|---|---|---|---|---|---|
| `000_synchronized_band_random` | +4.375 | `4.` | True | False | False |
| `001_synchronized_band_random` | -0.875 | `8.` | False | True | False |
| `002_synchronized_band_random` | +5.750 | `4.` | False | True | False |
| `003_synchronized_band_random` | +8.688 | `4.` | False | True | False |
| `004_synchronized_band_random` | +3.750 | `6.` | False | True | False |
| `005_synchronized_band_random` | -0.875 | `8.` | False | True | False |
| `006_synchronized_band_random` | +6.125 | `4.` | False | True | False |
| `007_synchronized_band_random` | +1.250 | `6.` | False | True | False |
| `008_synchronized_band_random` | +3.125 | `6.` | False | True | False |

### random_ant: [result.json](../../out/2026-09-08_134200_matched-random-band-ant/result.json)

| condition | S_swap | first token | donor words | spider | loop phrase |
|---|---|---|---|---|---|
| `000_synchronized_band_random` | +5.250 | `6.` | True | False | False |
| `001_synchronized_band_random` | -0.250 | `8.` | False | True | False |
| `002_synchronized_band_random` | +5.375 | `6.` | False | True | False |
| `003_synchronized_band_random` | +2.750 | `8.` | False | True | False |
| `004_synchronized_band_random` | +3.625 | `8.` | False | True | False |
| `005_synchronized_band_random` | -0.250 | `8.` | False | True | False |
| `006_synchronized_band_random` | -2.375 | `8.` | False | True | False |
| `007_synchronized_band_random` | +2.750 | `8.` | False | True | True |
| `008_synchronized_band_random` | +0.000 | `8.` | False | True | False |

## What is missing (do not guess)

- Exact rendered input IDs and historical exact-input provenance: not stored; not reconstructed.
- Readout calibration: no mapping from readout fragments to concepts; all readouts unvalidated.
- Independent validation set: selected development conditions only.
- Reliability across prompts: not shown; naming corrections and the dog-property stay-spider run are counter-evidence.